#### **Load Python packages**

In [ ]:
#!pip -q install openai faiss-cpu pymupdf tiktoken jsonschema tenacity

In [ ]:
import os, json, re, math, time, uuid, hashlib, pathlib, dataclasses, textwrap
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
import numpy as np
import pandas as pd
import faiss
import fitz  # PyMuPDF
from tenacity import retry, stop_after_attempt, wait_exponential
import tiktoken
from openai import OpenAI

RUN_ID = time.strftime('%Y%m%d-%H%M%S')
RUN_DIR = f'runs/{RUN_ID}'
os.makedirs(RUN_DIR, exist_ok=True)
os.makedirs('cache/embeddings', exist_ok=True)
os.makedirs('cache/responses', exist_ok=True)
print('RUN_ID =', RUN_ID)

RUN_ID = 20250920-132248


#### **Global Settings & Login to OpenAI**

- `STRICT_SCHEMA`: When set to `True`, the schema validation will be strictly enforced, raising errors for any discrepancies. When set to `False` (default), the model first stores the raw info and then tries to validate afterwards.
- `MODEL`: LLM model to use.
- `EMBED_MODEL`: Embedding model to use.
- `TOPK_`: Number of top hits to return from retrieval.

In [ ]:
# --- LOGIN TO OPENAI ---
import os, getpass
from openai import OpenAI
if not os.environ.get('OPENAI_API_KEY'):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter OPENAI_API_KEY: ")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# --- GLOBAL SETTINGS ---
STRICT_SCHEMA = False
MODEL = 'gpt-4o-mini'
EMBED_MODEL = 'text-embedding-3-small'
TOPK_CODEBOOK = 5
TOPK_DOC = 3

print('Config OK:', {'STRICT_SCHEMA': STRICT_SCHEMA, 'MODEL': MODEL, 'EMBED_MODEL': EMBED_MODEL})

Config OK: {'STRICT_SCHEMA': False, 'MODEL': 'gpt-4o-mini', 'EMBED_MODEL': 'text-embedding-3-small'}


#### **Set up hashing for caching**

In [ ]:
def sha256(s: str) -> str:
    return hashlib.sha256(s.encode('utf-8')).hexdigest()

class DiskCache:
    def __init__(self, base_dir: str):
        self.base = pathlib.Path(base_dir)
        self.base.mkdir(parents=True, exist_ok=True)
    def get(self, key: str) -> Optional[dict]:
        p = self.base / f'{key}.json'
        if p.exists():
            with open(p, 'r', encoding='utf-8') as f:
                return json.load(f)
        return None
    def set(self, key: str, obj: dict):
        p = self.base / f'{key}.json'
        with open(p, 'w', encoding='utf-8') as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)

RESP_CACHE = DiskCache('cache/responses')
EMB_CACHE = DiskCache('cache/embeddings')

class Tracer:
    def __init__(self, run_dir: str):
        self.run_dir = pathlib.Path(run_dir)
        self.trace_path = self.run_dir / 'trace.jsonl'
        self.run_dir.mkdir(parents=True, exist_ok=True)
    def log(self, event: dict):
        event['ts'] = time.time()
        with open(self.trace_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(event, ensure_ascii=False) + '\n')
        return event

TR = Tracer(RUN_DIR)
TR.log({'event':'run_start','run_id':RUN_ID})
print('Tracer ready →', RUN_DIR)

Tracer ready → runs/20250920-125710


#### **Set up working directory and file location.**

- For convience, simply adjust the `CODE_MODE` below (whatever it is right now) to `CODE_MODE= "manual_upload"` and then upload the files manually.

- In order to make the code to successfully detect the pdf, please rename the codebook exactly as 4ptCodebook.pdf and the research article as sample_paper.pdf


In [ ]:
CODE_MODE = "local"  # "colab", "local", or "manual_upload"

if CODE_MODE == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    PAPER_PDF = "/content/drive/MyDrive/AI44PT/data/processed/sample_paper.pdf"
    CODEBOOK_PDF = "/content/drive/MyDrive/AI44PT/data/processed/4ptCodebook.pdf"
elif CODE_MODE == "local":
    import os
    import pathlib
    cwd = os.getcwd()
    cwd_path = pathlib.Path(cwd)
    cwd = str(cwd_path.parent)
    print(f"Current working directory: {cwd}")
    PAPER_PDF = os.path.join(cwd, "data/processed/sample_paper.pdf")
    CODEBOOK_PDF = os.path.join(cwd, "data/processed/4ptCodebook.pdf")
elif CODE_MODE == "manual_upload":
    from google.colab import files
    print('Upload your PDFs. Please name the codebook file "4ptCodebook.pdf" and the paper file "sample_paper.pdf".')
    uploaded = files.upload()  # choose files in the UI
    print('Files uploaded:', list(uploaded.keys()))
    CODEBOOK_FILENAME = "4ptCodebook.pdf"
    DOC_FILENAME = "sample_paper.pdf"

    if CODEBOOK_FILENAME in uploaded and DOC_FILENAME in uploaded:
        CODEBOOK_PDF = CODEBOOK_FILENAME
        PAPER_PDF = DOC_FILENAME
        print(f"Codebook file: {CODEBOOK_PDF}, Document file: {PAPER_PDF}")
    else:
        print(f"Error: Please ensure you upload both '{CODEBOOK_FILENAME}' and '{DOC_FILENAME}'.")
        CODEBOOK_PDF = None
        PAPER_PDF = None

Current working directory: /Users/xinby/Library/CloudStorage/GoogleDrive-letheon.by@gmail.com/My Drive/AI44PT


#### **Set up PDF processing utils**
- Here we use `PyMuPDF` to process pdf

In [ ]:
def read_pdf(path: str):
    doc = fitz.open(path)
    pages = []
    for i, pg in enumerate(doc):
        pages.append({'page': i+1, 'text': pg.get_text('text') or ''})
    return pages

def split_chunks(pages, max_chars=2000, overlap=200):
    chunks, buf, buf_pages = [], '', []
    for p in pages:
        t = p['text']
        i = 0
        while i < len(t):
            take = t[i:i+(max_chars-len(buf))]
            buf += take
            if p['page'] not in buf_pages:
                buf_pages.append(p['page'])
            i += len(take)
            if len(buf) >= max_chars or i >= len(t):
                chunks.append({'id': str(uuid.uuid4()), 'text': buf.strip(), 'meta': {'pages': buf_pages.copy(), 'tag': None}})
                if overlap>0:
                    buf = buf[-overlap:]
                    buf_pages = buf_pages[-1:]
                else:
                    buf, buf_pages = '', []
    return chunks

cb_pages = read_pdf(CODEBOOK_PDF)
paper_pages = read_pdf(PAPER_PDF)
cb_chunks = split_chunks(cb_pages, max_chars=2000, overlap=200)
paper_chunks = split_chunks(paper_pages, max_chars=1800, overlap=200)
print('Chunks → codebook:', len(cb_chunks), 'document:', len(paper_chunks))

# Optional: Tag codebook chunks based on page ranges
CODEBOOK_TAG_RULES = {
    # 'DEFS': [(1,5)],
    # 'RULES': [(6,12)],
    # 'EFFECTS': [(13,18)],
    # 'EXAMPLES': [(19,30)],
}
for ch in cb_chunks:
    pages = ch['meta']['pages']
    for tag, ranges in CODEBOOK_TAG_RULES.items():
        for a,b in ranges:
            if any(a<=p<=b for p in pages):
                ch['meta']['tag'] = tag
                break

TR.log({'event':'chunks_built','codebook_chunks':len(cb_chunks),'paper_chunks':len(paper_chunks)})

Chunks → codebook: 130 document: 65


{'event': 'chunks_built',
 'codebook_chunks': 130,
 'paper_chunks': 65,
 'ts': 1758345399.426614}

#### **Set up embedding model**

In [ ]:
@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=1, max=8))
def embed(texts: List[str]):
    cache_key = sha256(EMBED_MODEL + '|' + str(len(texts)) + '|' + sha256('\n'.join(texts)[:2048]))
    cached = EMB_CACHE.get(cache_key)
    if cached: return np.array(cached['vecs'], dtype='float32')
    res = client.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = [d.embedding for d in res.data]
    EMB_CACHE.set(cache_key, {'vecs': vecs})
    return np.array(vecs, dtype='float32')

class FaissIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        self.vecs = None
        self.index = None
    def build(self):
        texts = [c['text'] for c in self.chunks]
        self.vecs = embed(texts)
        faiss.normalize_L2(self.vecs)
        self.index = faiss.IndexFlatIP(self.vecs.shape[1])
        self.index.add(self.vecs)
        return self
    def query(self, q: str, k: int=5, tag_filter: Optional[List[str]]=None):
        qv = embed([q])
        faiss.normalize_L2(qv)
        D, I = self.index.search(qv, k*3)  # 先多取，再按tag过滤
        hits = []
        for score, idx in zip(D[0], I[0]):
            if idx < 0: continue
            ch = self.chunks[idx]
            tag = ch['meta'].get('tag')
            if tag_filter and tag not in tag_filter:
                continue
            hits.append({'text': ch['text'], 'pages': ch['meta'].get('pages', []), 'tag': tag, 'score': float(score)})
            if len(hits) >= k: break
        return hits

cb_index = FaissIndex(cb_chunks).build()
doc_index = FaissIndex(paper_chunks).build()
TR.log({'event':'index_built','codebook_vecs':len(cb_index.chunks),'doc_vecs':len(doc_index.chunks)})
print('Indexes built.')

Indexes built.


#### ❗ **Question Bank**

- `id`: ID of the question
- `text`: The actual question text
- `kb_tags`: When specified, the model will prioritize searching in the codebook chunks with relevant tags. If empty, the model will search across all chunks.
- `doc_top_k` / `cb_top_k`: Number of top chunks to retrieve from the document / codebook respectively.

In [ ]:
QUESTION_BANK = [
  {"id": "Q1", "text": "Does the article fit in the universe of sustainability analyses we seek to assess? (Yes / No)", "kb_tags": []},
  {"id": "Q2", "text": "What problems or set of problems is the article trying to address?", "kb_tags": []},
  {"id": "Q3", "text": "Do the analysis, conclusions, and theories derived from, and directed to, understanding and/or managing a clearly specified \"on the ground\" problem or class of problems? (Yes / No)", "kb_tags": []},
  {"id": "Q4", "text": "Provide arguments that supports your response to Q3 (on the ground problem context). Plus, cite some key text from the article that supports your response.", "kb_tags": []},
  {"id": "Q5", "text": "Are the analysis, conclusions, and theories generated to apply beyond understanding, and/or managing a clearly specified “on the ground” problem or class of problems? (Yes / No)", "kb_tags": []},
  {"id": "Q6", "text": "Provide arguments that supports your response to Q5 (beyond on the ground problem context). Plus, cite some key text from the article that supports your response.", "kb_tags": []},
  {"id": "Q7", "text": "Do the analysis, conclusions, and theories treat individuals, organizations and states as largely self-interested, satisfaction driven entities that seek to maximize some kind “utility” outcome? (Yes / No)", "kb_tags": []},
  {"id": "Q8", "text": "Provide arguments that supports your response to Q7 (self-interested agents). Plus, cite some key text from the article that supports your response.", "kb_tags": []},
  {"id": "Q9", "text": "Do the analysis incorporate theories and conclusions incorporate an assessment of individuals, organizations and/or states that extends beyond self-interested satisfaction seeking motivations? (Yes / No)", "kb_tags": []},
  {"id": "Q10", "text": "Provide arguments that supports your response to Q9 (beyond self-interested agents). Plus, cite some key text from the article that supports your response.", "kb_tags": []},
  {"id": "Q11", "text": "Based on your answers to Q1-Q10, which 4PT Type (T1/T2/T3/T4) best characterizes the article? Provide a ≤120-char justification.", "kb_tags": []},
  {"id": "Q12", "text": "Do you detect any 'whack-a-mole' risk or solving T4 with T1/2/3 thinking? Cite 0–2 evidence spans.", "kb_tags": []}]

with open(f'{RUN_DIR}/question_bank.json', 'w', encoding='utf-8') as f:
    json.dump(QUESTION_BANK, f, ensure_ascii=False, indent=2)
print('Question bank saved at', f'{RUN_DIR}/question_bank.json')

Question bank saved at runs/20250920-132248/question_bank.json


#### **RAG function to answer questions**

In [ ]:
def format_hits(hits, title):
    lines = [f'## {title}']
    for i,h in enumerate(hits,1):
        pg = ','.join(map(str,h['pages'])) if h['pages'] else 'NA'
        snippet = h['text'].strip().replace('\n',' ')
        if len(snippet)>700: snippet = snippet[:700] + '…'
        tag = f"[{h['tag']}]" if h.get('tag') else ''
        lines.append(f"[{i}] (p.{pg}){tag} score={h['score']:.3f} | {snippet}")
    return '\n'.join(lines)

def compose_prompt(doc_id: str, q: dict, cb_hits, doc_hits):
    guide = textwrap.dedent(f"""
    You are an expert 4PT analyst. Use ONLY the provided **Codebook** context and **Document Evidence**.
    - Cite short evidence (with page numbers) when asserting.
    - If evidence is missing, say so explicitly (do NOT invent).
    - Keep answers concise. For justifications, ≤120 characters.
    """
    ).strip()
    ctx = "\n\n".join([
        format_hits(cb_hits, '4PT Codebook Context'),
        format_hits(doc_hits, 'Document Evidence')
    ])
    prompt = f"""
{guide}

{ctx}

### Question {q['id']}
{q['text']}
    """.strip()
    return prompt

def retrieve_for_q(q: dict):
    kb_topk = q.get('kb_topk', TOPK_CODEBOOK)
    doc_topk = q.get('doc_topk', TOPK_DOC)
    cb_hits = cb_index.query(q['text'], k=kb_topk, tag_filter=q.get('kb_tags'))
    doc_hits = doc_index.query(q['text'], k=doc_topk)
    return cb_hits, doc_hits

#### Model call to answer questions

In [ ]:
def call_llm_with_cache(model: str, prompt: str, qid: str):
    key = sha256(model + '|' + prompt)
    cached = RESP_CACHE.get(key)
    out_dir = pathlib.Path(RUN_DIR) / f'q_{qid}'
    out_dir.mkdir(parents=True, exist_ok=True)

    if cached:
        with open(out_dir/'response_raw.txt','w',encoding='utf-8') as f:
            f.write(cached['text'])
        if 'usage' in cached:
            with open(out_dir/'usage.json','w',encoding='utf-8') as f:
                json.dump(cached['usage'], f, ensure_ascii=False, indent=2)
        TR.log({'event':'llm_cache_hit','qid':qid,'model':model,'cache_key':key})
        return cached['text'], cached.get('usage'), True

    t0 = time.time()
    resp = client.responses.create(model=model, input=prompt)
    dt = time.time()-t0
    text = resp.output_text
    usage = getattr(resp,'usage',None)
    usage_dict = usage.dict() if usage else {}
    TR.log({'event':'llm_call','qid':qid,'model':model,'latency_s':dt,'usage':usage_dict})

    with open(out_dir/'response_raw.txt','w',encoding='utf-8') as f:
        f.write(text)
    with open(out_dir/'usage.json','w',encoding='utf-8') as f:
        json.dump(usage_dict, f, ensure_ascii=False, indent=2)
    RESP_CACHE.set(key, {'text': text, 'usage': usage_dict})
    return text, usage_dict, False

In [ ]:
DOC_ID = f'doc-{RUN_ID}'

def run_pipeline(doc_id: str, question_bank: List[Dict[str,Any]]):
    summary_rows = []
    for q in question_bank:
        qid = q['id']
        qdir = pathlib.Path(RUN_DIR) / f'q_{qid}'
        qdir.mkdir(parents=True, exist_ok=True)
        cb_hits, doc_hits = retrieve_for_q(q)
        with open(qdir/'contexts.json','w',encoding='utf-8') as f:
            json.dump({'codebook': cb_hits, 'document': doc_hits}, f, ensure_ascii=False, indent=2)
        prompt = compose_prompt(doc_id, q, cb_hits, doc_hits)
        with open(qdir/'prompt.txt','w',encoding='utf-8') as f:
            f.write(prompt)
        text, usage, cache_hit = call_llm_with_cache(MODEL, prompt, qid)
        row = {
            'run_id': RUN_ID,
            'doc_id': doc_id,
            'qid': qid,
            'model': MODEL,
            'cache_hit': cache_hit,
            'prompt_tokens': usage.get('prompt_tokens') if usage else None,
            'completion_tokens': usage.get('completion_tokens') if usage else None,
            'total_tokens': usage.get('total_tokens') if usage else None,
        }
        summary_rows.append(row)
        TR.log({'event':'q_done','qid':qid, 'cache_hit': cache_hit, 'usage': usage})
    df = pd.DataFrame(summary_rows)
    df.to_csv(f'{RUN_DIR}/summary.csv', index=False)
    TR.log({'event':'run_done','questions':len(question_bank)})
    return df

df_summary = run_pipeline(DOC_ID, QUESTION_BANK)
df_summary

,run_id,doc_id,qid,model,cache_hit,prompt_tokens,completion_tokens,total_tokens
0,20250920-132248,doc-20250920-132248,Q1,gpt-4o-mini,False,None,None,1475
1,20250920-132248,doc-20250920-132248,Q2,gpt-4o-mini,False,None,None,1474
2,20250920-132248,doc-20250920-132248,Q3,gpt-4o-mini,False,None,None,1444
3,20250920-132248,doc-20250920-132248,Q4,gpt-4o-mini,False,None,None,1551
4,20250920-132248,doc-20250920-132248,Q5,gpt-4o-mini,False,None,None,1422
5,20250920-132248,doc-20250920-132248,Q6,gpt-4o-mini,False,None,None,1540
6,20250920-132248,doc-20250920-132248,Q7,gpt-4o-mini,False,None,None,1481
7,20250920-132248,doc-20250920-132248,Q8,gpt-4o-mini,False,None,None,1765
8,20250920-132248,doc-20250920-132248,Q9,gpt-4o-mini,False,None,None,1450
9,20250920-132248,doc-20250920-132248,Q10,gpt-4o-mini,False,None,None,1544


In [ ]:
from IPython.display import display, HTML
import pathlib, json

def show_run_dashboard(run_dir: str):
    runp = pathlib.Path(run_dir)
    df = pd.read_csv(runp/'summary.csv') if (runp/'summary.csv').exists() else pd.DataFrame()
    display(df)
    for q in QUESTION_BANK:
        qid = q['id']
        qdir = runp / f'q_{qid}'
        print('\n','='*5, f'Question {qid}: {q["text"]}', '='*5)
        ctx = json.load(open(qdir/'contexts.json','r',encoding='utf-8'))
        print('\n[Codebook Hits]')
        for h in ctx['codebook']:
            print(f"p{h['pages']} tag={h.get('tag')} score={h['score']:.3f} → {h['text'][:160].replace('\n',' ')}…")
        print('\n[Document Hits]')
        for h in ctx['document']:
            print(f"p{h['pages']} score={h['score']:.3f} → {h['text'][:160].replace('\n',' ')}…")
        print('\n[Prompt]\n')
        print(open(qdir/'prompt.txt','r',encoding='utf-8').read()[:1800])
        print('\n[Model Response]\n')
        print(open(qdir/'response_raw.txt','r',encoding='utf-8').read()[:1800])
        if (qdir/'usage.json').exists():
            print('\n[Usage]\n', json.load(open(qdir/'usage.json','r',encoding='utf-8')))

show_run_dashboard(RUN_DIR)

,run_id,doc_id,qid,model,cache_hit,prompt_tokens,completion_tokens,total_tokens
0,20250920-132248,doc-20250920-132248,Q1,gpt-4o-mini,False,NaN,NaN,1475
1,20250920-132248,doc-20250920-132248,Q2,gpt-4o-mini,False,NaN,NaN,1474
2,20250920-132248,doc-20250920-132248,Q3,gpt-4o-mini,False,NaN,NaN,1444
3,20250920-132248,doc-20250920-132248,Q4,gpt-4o-mini,False,NaN,NaN,1551
4,20250920-132248,doc-20250920-132248,Q5,gpt-4o-mini,False,NaN,NaN,1422
5,20250920-132248,doc-20250920-132248,Q6,gpt-4o-mini,False,NaN,NaN,1540
6,20250920-132248,doc-20250920-132248,Q7,gpt-4o-mini,False,NaN,NaN,1481
7,20250920-132248,doc-20250920-132248,Q8,gpt-4o-mini,False,NaN,NaN,1765
8,20250920-132248,doc-20250920-132248,Q9,gpt-4o-mini,False,NaN,NaN,1450
9,20250920-132248,doc-20250920-132248,Q10,gpt-4o-mini,False,NaN,NaN,1544



 ===== Question Q1: Does the article fit in the universe of sustainability analyses we seek to assess? (Yes / No) =====

[Codebook Hits]
p[39, 40] tag=None score=0.567 → thor’s graduating  university (highest degree)  Text  14  Corresponding author's geographic  location (Highest degree)    15  Corresponding author's PhD  disser…
p[31, 32] tag=None score=0.551 → ansparency” among transnational sustainability governors. This limits the scope for transparency- induced accountability in this policy domain.  Comments from p…
p[11] tag=None score=0.548 → orcing); Optimization (Type 2 reinforcing);  Compromise (Type 3 reinforcing) and Prioritization (Type 4 reinforcing). This broad treatment that there  can, and …
p[32, 33] tag=None score=0.544 → ntirely a bad thing if the legality regime overtook sustainability certification. Behind these  specific arguments are general perspectives on how domestic circ…
p[28] tag=None score=0.532 → ude ambiguous cases and or cases beyond our universe (i.e